# Исследование надежности заемщиков

Это первая часть проекта, она будет проверена автоматически. Вторую часть проверит ревьюер.

## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [85]:
# импортируйте библиотеку pandas
import pandas as pd

In [86]:
# Читаем файл прямо из корневой папки Colab
data = pd.read_csv('data.csv')

# Проверяем, что всё загрузилось
data.head()# прочитайте csv-файл

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу


**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [87]:
print(data.head(20))# ваш код здесь

    children  days_employed  dob_years            education  education_id  \
0          1   -8437.673028         42               высшее             0   
1          1   -4024.803754         36              среднее             1   
2          0   -5623.422610         33              Среднее             1   
3          3   -4124.747207         32              среднее             1   
4          0  340266.072047         53              среднее             1   
5          0    -926.185831         27               высшее             0   
6          0   -2879.202052         43               высшее             0   
7          0    -152.779569         50              СРЕДНЕЕ             1   
8          2   -6929.865299         35               ВЫСШЕЕ             0   
9          0   -2188.756445         41              среднее             1   
10         2   -4171.483647         36               высшее             0   
11         0    -792.701887         40              среднее             1   

**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [88]:
data.info()# ваш код здесь

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [89]:
print(data.isna().sum())# ваш код здесь

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64


**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [90]:
# ваш код здесь
#data['days_employed'] = data['days_employed'].fillna(0)
data['total_income'] = data['total_income'].fillna(
    data.groupby('income_type')['total_income'].transform('median')
)
print(data.head(15))


    children  days_employed  dob_years            education  education_id  \
0          1   -8437.673028         42               высшее             0   
1          1   -4024.803754         36              среднее             1   
2          0   -5623.422610         33              Среднее             1   
3          3   -4124.747207         32              среднее             1   
4          0  340266.072047         53              среднее             1   
5          0    -926.185831         27               высшее             0   
6          0   -2879.202052         43               высшее             0   
7          0    -152.779569         50              СРЕДНЕЕ             1   
8          2   -6929.865299         35               ВЫСШЕЕ             0   
9          0   -2188.756445         41              среднее             1   
10         2   -4171.483647         36               высшее             0   
11         0    -792.701887         40              среднее             1   

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. Таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [91]:
# ваш код здесь
data['days_employed'] = abs(data['days_employed'])
print(data.head(15))


    children  days_employed  dob_years            education  education_id  \
0          1    8437.673028         42               высшее             0   
1          1    4024.803754         36              среднее             1   
2          0    5623.422610         33              Среднее             1   
3          3    4124.747207         32              среднее             1   
4          0  340266.072047         53              среднее             1   
5          0     926.185831         27               высшее             0   
6          0    2879.202052         43               высшее             0   
7          0     152.779569         50              СРЕДНЕЕ             1   
8          2    6929.865299         35               ВЫСШЕЕ             0   
9          0    2188.756445         41              среднее             1   
10         2    4171.483647         36               высшее             0   
11         0     792.701887         40              среднее             1   

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [92]:
# ваш код здесь
income_type_gr = data.groupby('income_type')['days_employed'].median()
print(income_type_gr)

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64


У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [93]:
# ваш код здесь
print(data['children'].unique())

[ 1  0  3  2 -1  4 20  5]


**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [94]:
# ваш код здесь
data = data[(data['children']!= -1)&(data['children']!= 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [95]:
# ваш код здесь
print(data['children'].unique())

[1 0 3 2 4 5]


### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждому типу занятости `income_type`.**

In [96]:
# ваш код здесь
data['days_employed'] = data['days_employed'].fillna(
    data.groupby('income_type')['days_employed'].transform('median')
)
print(data.head(15))

    children  days_employed  dob_years            education  education_id  \
0          1    8437.673028         42               высшее             0   
1          1    4024.803754         36              среднее             1   
2          0    5623.422610         33              Среднее             1   
3          3    4124.747207         32              среднее             1   
4          0  340266.072047         53              среднее             1   
5          0     926.185831         27               высшее             0   
6          0    2879.202052         43               высшее             0   
7          0     152.779569         50              СРЕДНЕЕ             1   
8          2    6929.865299         35               ВЫСШЕЕ             0   
9          0    2188.756445         41              среднее             1   
10         2    4171.483647         36               высшее             0   
11         0     792.701887         40              среднее             1   

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [97]:
# ваш код здесь
print(data.isna().sum())

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64


### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [98]:
# ваш код здесь
data['total_income'] = data['total_income'].astype('int')


### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру.**

In [99]:
# ваш код здесь
data['education'] = data['education'].str.lower()


**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [100]:
# посчитайте дубликаты
data.duplicated().count()


np.int64(21402)

In [101]:
# удалите дубликаты
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [102]:
# создайте функцию categorize_income()
def categorize_income(income):

    if income <= 30000:
        return 'E'
    if income <= 50000:
        return 'D'
    if income <= 200000:
        return 'C'
    if income <= 1000000:
        return 'B'
    if income > 1000001:
        return 'A'





In [103]:
# примените функцию методом apply()
data['total_income_category'] = data['total_income'].apply(categorize_income)
print(data.head(15))

    children  days_employed  dob_years            education  education_id  \
0          1    8437.673028         42               высшее             0   
1          1    4024.803754         36              среднее             1   
2          0    5623.422610         33              среднее             1   
3          3    4124.747207         32              среднее             1   
4          0  340266.072047         53              среднее             1   
5          0     926.185831         27               высшее             0   
6          0    2879.202052         43               высшее             0   
7          0     152.779569         50              среднее             1   
8          2    6929.865299         35               высшее             0   
9          0    2188.756445         41              среднее             1   
10         2    4171.483647         36               высшее             0   
11         0     792.701887         40              среднее             1   

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [104]:
# ваш код здесь
print(data['purpose'].unique())
print(data['purpose'].value_counts())

['покупка жилья' 'приобретение автомобиля' 'дополнительное образование'
 'сыграть свадьбу' 'операции с жильем' 'образование'
 'на проведение свадьбы' 'покупка жилья для семьи' 'покупка недвижимости'
 'покупка коммерческой недвижимости' 'покупка жилой недвижимости'
 'строительство собственной недвижимости' 'недвижимость'
 'строительство недвижимости' 'на покупку подержанного автомобиля'
 'на покупку своего автомобиля' 'операции с коммерческой недвижимостью'
 'строительство жилой недвижимости' 'жилье'
 'операции со своей недвижимостью' 'автомобили' 'заняться образованием'
 'сделка с подержанным автомобилем' 'получение образования' 'автомобиль'
 'свадьба' 'получение дополнительного образования' 'покупка своего жилья'
 'операции с недвижимостью' 'получение высшего образования'
 'свой автомобиль' 'сделка с автомобилем' 'профильное образование'
 'высшее образование' 'покупка жилья для сдачи' 'на покупку автомобиля'
 'ремонт жилью' 'заняться высшим образованием']
purpose
свадьба              

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [105]:
# создайте функцию categorize_purpose()
def categorize_purpose(q):
    if 'жил' in q:
        return 'операции с недвижимостью'
    if 'недвиж' in q:
        return 'операции с недвижимостью'
    if 'свадьб' in q:
        return 'проведение свадьбы'
    if 'авто' in q:
        return 'операции с автомобилем'
    if 'образ' in q:
        return 'получение образования'


In [106]:
# примените функцию методом apply()

data['purpose_category'] = data['purpose'].apply(categorize_purpose)
data['purpose_category'].value_counts()

,count
purpose_category,
операции с недвижимостью,10751
операции с автомобилем,4279
получение образования,3988
проведение свадьбы,2313


**Задание 19. Есть ли зависимость между количеством детей и возвратом кредита в срок?**

In [107]:
# Агрегация данных в одну строку с помощью .agg()
debt_children = data.groupby('children')['debt'].agg(['count', 'sum', 'mean'])

# Переименуем колонки для красоты (mean - это и есть конверсия в должника, просто умножим на 100)
debt_children.columns = ['Всего заемщиков', 'Кол-во должников', 'Доля просрочки (%)']
debt_children['Доля просрочки (%)'] = round(debt_children['Доля просрочки (%)'] * 100, 2)

# Сортируем по доле просрочки
print(debt_children.sort_values('Доля просрочки (%)'))

          Всего заемщиков  Кол-во должников  Доля просрочки (%)
children                                                       
5                       9                 0                0.00
0                   14091              1063                7.54
3                     330                27                8.18
1                    4808               444                9.23
2                    2052               194                9.45
4                      41                 4                9.76


#### Как мы видим из таблицы большая часть кредиторов бездетные и у них самый лучший показатель по возврату кредита в срок (с 4 и 5 детьми учитывать не можем, так как выборка всего по 9 и соотвественно 41 заемщикам, эта выборка слишком мала для анализа, нужно больше данных). Следующие по большинству после бездетных идут заемщики с одним ребенком, но они возврвщают хуже чем заемщики с 3 детьми, но лучше чем 2. Очень неодназначный данные для анализа зависимости количества детей на возврат кредита в срок.

In [108]:
# Выведите на экран перечень уникальных значений из столбца family_status
print(data['family_status'].unique())
data['family_status'] = data['family_status'].str.lower()

['женат / замужем' 'гражданский брак' 'вдовец / вдова' 'в разводе'
 'Не женат / не замужем']


#### Задание 20. Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [109]:
## Агрегация данных по семейному положению
debt_family = data.groupby('family_status')['debt'].agg(['count', 'sum', 'mean'])

# Переименование колонок для бизнес-визуализации
debt_family.columns = ['Всего заемщиков', 'Кол-во должников', 'Доля просрочки (%)']

# Перевод доли в проценты и округление до 2 знаков
debt_family['Доля просрочки (%)'] = round(debt_family['Доля просрочки (%)'] * 100, 2)

# Вывод отсортированной таблицы
print(debt_family.sort_values('Доля просрочки (%)'))

                       Всего заемщиков  Кол-во должников  Доля просрочки (%)
family_status                                                               
вдовец / вдова                     951                63                6.62
в разводе                         1189                84                7.06
женат / замужем                  12261               927                7.56
гражданский брак                  4134               385                9.31
не женат / не замужем             2796               273                9.76


#### По данным таблицы прослеживается взаимосвязь между семейным статусом и возвратом кредита в срок. Заемщики, которые были в браке(вдовец / вдова и в в разводе), а также состоят в официальном браке реже просрачиваю выплаты по кредитам в срок по сравнению с теми, кто не состоит в официально зарегестрированном браке(гражданский брак, не женат / не замужем).

#### Задание 21. Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [110]:
# Ранее мы систематизировали доходы заемщиков методом добавления нового столбца столбец total_income_category с категориями:

#0–30000 — 'E';
#30001–50000 — 'D';
#50001–200000 — 'C';
#200001–1000000 — 'B';
#1000001 и выше — 'A'.
# Выведете на экран в виде таблицы результат влияния дохода на возврат кредита в срок:
# Агрегация данных по категориям дохода (А - высший, Е - низший)
debt_income = data.groupby('total_income_category')['debt'].agg(['count', 'sum', 'mean'])

# Переименование колонок
debt_income.columns = ['Всего заемщиков', 'Кол-во должников', 'Доля просрочки (%)']

# Перевод доли в проценты и округление
debt_income['Доля просрочки (%)'] = round(debt_income['Доля просрочки (%)'] * 100, 2)

# Вывод отсортированной таблицы
print(debt_income.sort_values('Доля просрочки (%)'))


                       Всего заемщиков  Кол-во должников  Доля просрочки (%)
total_income_category                                                       
D                                  349                21                6.02
B                                 5014               354                7.06
A                                   25                 2                8.00
C                                15921              1353                8.50
E                                   22                 2                9.09


#### Как видим  заемщики с уровнем дохода 30001–50000 руб по данным таблицы имеют меньше просроков, чем те у кого уровень свыше 200 000 руб, но так как именно таких кредиторов большинстово(С - 50001–200000), то и выборка по ним в разы больше, по сравнению с другими категориями заемщиков. Поэтому ориентироваться на эти данные не стоит.

#### Задание 22. Как разные цели кредита влияют на его возврат в срок?

In [112]:
# Выведете на экран в виде таблицы результат влияния разных целей на возврат кредита в срок:
# Агрегация данных по целям кредита
debt_purpose = data.groupby('purpose_category')['debt'].agg(['count', 'sum', 'mean'])

# Переименование колонок
debt_purpose.columns = ['Всего заемщиков', 'Кол-во должников', 'Доля просрочки (%)']

# Перевод доли в проценты и округление
debt_purpose['Доля просрочки (%)'] = round(debt_purpose['Доля просрочки (%)'] * 100, 2)

# Вывод отсортированной таблицы
print(debt_purpose.sort_values('Доля просрочки (%)'))



                          Всего заемщиков  Кол-во должников  \
purpose_category                                              
операции с недвижимостью            10751               780   
проведение свадьбы                   2313               183   
получение образования                3988               369   
операции с автомобилем               4279               400   

                          Доля просрочки (%)  
purpose_category                              
операции с недвижимостью                7.26  
проведение свадьбы                      7.91  
получение образования                   9.25  
операции с автомобилем                  9.35  


#### По данным таблицы видим, что чаще всего кредит берут на операции с недвижимостью и эта же категория заемщиков имеет меньше всего просроков по  редиту, следующая категория заемщиков, которых меньше, а также чаще других возвращают кредиты в срок, чем остальные заемщики по цели кредита - это на проведени свадьбы. Заемщики, которые берут кредит на получение образования и операции с автомобилем, имеют больше просроков по кредиту по сравнению с другими категориями заемщиков и целями взятия кредита. Так что нужно учитывать цели кредита на его возврат в срок, так как прослеживается зависимость.

#### Задание 23. Приведите возможные причины появления пропусков в исходных данных.

#### В двух столбцах были пропущенные значения. Один из них — `days_employed`. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. Возможными причинами пропусков в исходных данных могли быть неправильное их заполнение, или заемщик пытался скрыть или исказить информацию, сбой при обработке данных.


**Задание 24. Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных**

#### На сумму дохода сильнее всего влияет тип занятости, поэтому заполняли пропуски в этом столбце  медианным значением, так как это позволяет недостающие данные заменять данными из соседних значений данного столбца, что позволяет лучше отобразить искаженные отсутсвующие данные.

### Общий вывод

«**Бизнес-вывод для скоринговой модели:**
> Проведен анализ исторических данных о выдаче кредитов для выявления факторов, влияющих на вероятность дефолта (Default Rate) заемщика.
> 1. **Портрет идеального заемщика (Низкий риск):** Клиенты, состоящие или состоявшие в официальном браке, без детей, берущие ипотеку (операции с недвижимостью). Доля просрочки в этих сегментах минимальна (~7.2 - 7.5%). Это обеспеченные залогом кредиты.
> 2. **Портрет рискованного заемщика (Высокий риск):** Клиенты, не состоящие в официальном браке, с 1-2 детьми, берущие кредит на покупку автомобиля или образование. Доля просрочки достигает 9.3 - 9.7%. Автокредиты и потребительские кредиты на образование несут повышенный риск невозврата.
>
> **Рекомендация бизнесу:** При настройке скоринговой модели рекомендуется ввести понижающие коэффициенты процентной ставки для ипотечных заемщиков в браке (для удержания надежной базы) и заложить дополнительную риск-маржу в ставку для одиноких клиентов с детьми при выдаче автокредитов.»
